# GT(gt_schoolnames.csv)와 대조 — school_candidate가 실제 학교명인지 검증

`count.ipynb`가 저장한 `data/processed/school_candidate_counts.csv`를 입력으로 사용.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src"))

REPO_ROOT

In [ ]:
import pandas as pd

df = pd.read_csv(
    REPO_ROOT / "data" / "processed" / "school_candidate_counts.csv", encoding="utf-8-sig"
)
df["school_candidate"] = df["school_candidate"].fillna("")  # 후보 0건인 행은 NaN으로 읽히므로 방어
df.shape

## school_candidate를 접미사 확장해서 GT와 대조

In [ ]:
from preprocessing import expand_school_suffix

gt_df = pd.read_csv(REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv", encoding="utf-8-sig")
gt_names = set(gt_df["학교명"])
# 약어 컬럼은 build_abbreviations.ipynb를 돌린 뒤에만 생김 (아직이면 빈 set이어도 정상)
gt_aliases = set(gt_df["약어"].dropna()) if "약어" in gt_df.columns else set()


def gt_match(candidate_text: str) -> str:
    """school_candidate의 각 토큰을 GT 약어와 먼저 대조하고, 안 되면 접미사 확장해서 정식명과 대조."""
    matched = []
    for tok in candidate_text.split():
        if tok in gt_aliases:
            matched.append(tok)
            continue
        for expanded in expand_school_suffix(tok):
            if expanded in gt_names:
                matched.append(expanded)
                break
    return " ".join(matched)


df["gt_match"] = df["school_candidate"].apply(gt_match)
df["gt_match_count"] = df["gt_match"].apply(lambda s: len(s.split()) if s else 0)
df[["comment", "school_candidate", "gt_match", "gt_match_count"]].head(20)

In [ ]:
# school_candidate_count 대비 실제 GT에 매칭된 개수 분포
# (예: count=1인데 gt_match_count=0 -> "연대"처럼 약칭이라 GT엔 없는 케이스)
df["gt_match_count"].value_counts().sort_index()

## GT 매칭 실패 케이스 — 약어 후보 vs 오검출(일반 명사) 구분 필요

In [ ]:
mismatch_df = df[(df["school_candidate_count"] >= 1) & (df["gt_match_count"] == 0)]
print(f"{len(mismatch_df)}개 행")
mismatch_df[["comment_id", "comment", "school_candidate"]]

In [ ]:
out_path = REPO_ROOT / "data" / "processed" / "gt_match_results.csv"
df[
    [
        "comment_id",
        "comment",
        "comment_noun",
        "school_candidate",
        "school_candidate_count",
        "gt_match",
        "gt_match_count",
    ]
].to_csv(out_path, index=False, encoding="utf-8-sig")
out_path